## Parsing methods Experimentation

Quantitative Comparison Metrics:
- Accuracy in table and graphical extraction
- Ability to handle complex formatting (Images, tables, equations)
- Preservation of document structure
- Speed of parsing

Evaluation techniques:
- Manual visual inspection
- End-to-End Testing: Run the entire RAG pipeline with different parsing strategies and evaluating the final output.

---
ref: https://www.reddit.com/r/LangChain/comments/1ef12q6/the_rag_engineers_guide_to_document_parsing/


### Quick Start
1. Run cells 2 to 6 in order.
2. Start MLflow UI in terminal:
   - `mlflow ui --backend-store-uri /Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns`
3. Review parser artifacts in MLflow UI:
   - Experiment -> Individual runs -> Artifacts
4. Log manual inspection scores:
   - `log_manual_inspection_by_parser("pymupdf4llm", 4, "Good structure, minor table loss", run_ids)`
   - `log_manual_inspection_by_parser("docling", 5, "Best structure preservation", run_ids)`

In [7]:
import glob
import mlflow
import time
from pathlib import Path

#pdfs = glob.glob("../data/raw/*pdf")
#target_filepath = glob.glob("../data/raw/attention-is-all-you-need.pdf")[0]
#target_filepath

target_filepath = "../data/raw/attention-is-all-you-need-full.pdf"
target_file= Path(target_filepath).name
target_file

'attention-is-all-you-need-full.pdf'

In [8]:
mlflow.set_tracking_uri("file:///Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns")
# TODO: use sqlite instead of filestore -> mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Papermind_Parsing_Audit") 

<Experiment: artifact_location='file:///Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns/573178881146081418', creation_time=1777274304251, experiment_id='573178881146081418', last_update_time=1777274304251, lifecycle_stage='active', name='Papermind_Parsing_Audit', tags={}, trace_location=None, workspace='default'>

In [ ]:
import mlflow
import time
from pathlib import Path

def run_parsing_experiment(parser_name, pdf_path, parse_func, parser_version):
    """
    Standardized wrapper to run a parser and log results to MLflow.
    Returns the MLflow run_id so manual scoring can be done without copy-paste.
    """
    with mlflow.start_run(run_name=f"{parser_name}_{Path(pdf_path).stem}") as run:
        run_id = run.info.run_id

        # 1. Log Metadata
        mlflow.log_param("parser", parser_name)
        mlflow.log_param("file_name", Path(pdf_path).name)

        # 2. Execute & Time
        start_time = time.time()
        try:
            markdown_text = parse_func(pdf_path)
            duration = time.time() - start_time

            # 3. Log Performance Metrics
            mlflow.log_metric("latency_sec", round(duration, 2))
            mlflow.log_metric("char_count", len(markdown_text))
            mlflow.log_param("parser_version", parser_version)

            # 4. Save Artifact (The actual Markdown)
            output_file = f"experiments/outputs/{parser_name}_result.md"
            Path(output_file).parent.mkdir(parents=True, exist_ok=True)
            with open(output_file, "w") as f:
                f.write(markdown_text)
            mlflow.log_artifact(output_file)

            print(f"{parser_name} complete. Open {output_file} to inspect.")
            print(f"Run ID: {run_id}")

        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_text(str(e), "error_log.txt")
            print(f"{parser_name} failed: {e}")

        return run_id


def log_manual_inspection(run_id, score, comments):
    """
    Call this after you've looked at the output file.
    score: 1-5 (1=Trash, 5=Perfect)
    """
    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric("manual_visual_score", score)
        mlflow.set_tag("manual_inspection_notes", comments)
        print(f"Logged score {score} for run {run_id}")


def log_manual_inspection_by_parser(parser_name, score, comments, run_ids_dict):
    """Convenience helper: log manual score using parser name instead of run_id."""
    run_id = run_ids_dict.get(parser_name)
    if not run_id:
        raise ValueError(f"No run_id found for parser '{parser_name}'.")
    log_manual_inspection(run_id, score, comments)

In [ ]:
import pymupdf4llm
from docling.document_converter import DocumentConverter

def parse_with_pymupdf4llm(pdf_path: str) -> str:
    return pymupdf4llm.to_markdown(pdf_path)


def parse_with_docling(pdf_path: str) -> str:
    converter = DocumentConverter()
    doc = converter.convert(pdf_path).document
    return doc.export_to_markdown()

import subprocess
def parse_with_mineru(pdf_path: str) -> str:
    output_dir = Path("experiments/outputs/mineru")
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "mineru",
        "-p", pdf_path,
        "-o", str(output_dir),
        "--use_vlm",
    ]
    subprocess.run(cmd, check=True)

    # MinerU usually writes markdown files into the output directory
    md_files = list(output_dir.rglob("*.md"))
    if not md_files:
        raise FileNotFoundError(f"No markdown output found in {output_dir}")

    # Return the first markdown file found
    return md_files[0].read_text(encoding="utf-8")

run_ids = {}
run_ids["pymupdf4llm"] = run_parsing_experiment("pymupdf4llm", target_filepath, parse_with_pymupdf4llm, "1.27.2.2")
run_ids["docling"] = run_parsing_experiment("docling", target_filepath, parse_with_docling, "2.91.0")
run_ids["mineru"] = run_parsing_experiment("mineru", target_filepath, parse_with_mineru, "3.1.5")
print("Run IDs:", run_ids)

pymupdf4llm complete. Open experiments/outputs/pymupdf4llm_result.md to inspect.
Run ID: 0d6533f2983e44aa9cee21db0a8ff57b


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1314.39it/s]


docling complete. Open experiments/outputs/docling_result.md to inspect.
Run ID: a44b5ff59e5642719abce281c0b7480f


2026-04-27 18:01:16.709 | INFO     | mineru.cli.client:run_orchestrated_cli:874 - Started local mineru-api at http://127.0.0.1:61237
2026-04-27 18:01:18.069 | INFO     | __main__:create_app:260 - Request concurrency limited to 1
INFO:     Started server process [90336]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:61237 (Press CTRL+C to quit)


Start MinerU FastAPI Service: http://127.0.0.1:61237
API documentation: http://127.0.0.1:61237/docs


2026-04-27 18:01:18.727 | INFO     | mineru.cli.client:run_planned_task:771 - Submitting batch 1/1 | 1 document, 9 pages in this batch | 9 pages total | task#1 [attention-is-all-you-need-full]
2026-04-27 18:01:21.908 | INFO     | mineru.utils.engine_utils:get_vlm_engine:34 - Using mlx-engine as the inference engine for VLM.
Fetching 13 files: 100%|██████████| 13/13 [00:33<00:00,  2.55s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
2026-04-27 18:02:08.321 | INFO     | mineru.backend.vlm.vlm_analyze:get_model:251 - get mlx-engine predictor cost: 34.91s
2026-04-27 18:02:08.386 | INFO     | mineru.backend.hybrid.hybrid_analyze:aio_doc_analyze:706 -

mineru complete. Open experiments/outputs/mineru_result.md to inspect.
Run ID: 7cb5b4b6bad140bcb3a905e55c88b84c
Run IDs: {'pymupdf4llm': '0d6533f2983e44aa9cee21db0a8ff57b', 'docling': 'a44b5ff59e5642719abce281c0b7480f', 'mineru': '7cb5b4b6bad140bcb3a905e55c88b84c'}


### Update runs after manual inspection

In [13]:
# After manual review, log scores without copying run_id:
log_manual_inspection_by_parser("pymupdf4llm", 2, "images omitted. sections and equations are not preserved", run_ids)
log_manual_inspection_by_parser("docling", 4, "Good structure preservation, Tables are preserved. For images, only text in images are preserved. Equations are not preserved", run_ids)
log_manual_inspection_by_parser("mineru", 5, "Best so far. Preserves all images, tables and equations", run_ids)


Logged score 2 for run 0d6533f2983e44aa9cee21db0a8ff57b
Logged score 4 for run a44b5ff59e5642719abce281c0b7480f
Logged score 5 for run 7cb5b4b6bad140bcb3a905e55c88b84c


---
### Final - Save parsing output

In [12]:
import json

# Save as JSON
output_dir = f"../data/processed/{Path(target_file).stem}.json"
with open(output_dir, "w") as file:
    file.write(json_output)

# Save as md
output_dir = f"../data/processed/{Path(target_file).stem}.md"
with open(output_dir, "w", encoding="utf-8") as file:
    file.write(md_output)

NameError: name 'json_output' is not defined